# Gaussian GRAPE Stage 4: 2D Landscape Slices

Stage 4 evaluates the existing Gaussian objective on local 2D grids around representative optima selected in
Stage 3. It does **not** run new optimization.

For the original 8D Gaussian family, each slice varies one channel's `(A_m, width_m)` while holding the other
six parameters fixed.

For the richer 24D two-Gaussian movable-center family, each slice varies one selected channel and one selected
Gaussian component's `(A, width)` while holding that component's center and all other parameters fixed.

These plots are 2D slices through the full endpoint landscape. They visualize local geometry; they do not prove
the full number of optima.


## Simulation Prerequisites

The following cells reproduce the existing Gaussian physics/objective plumbing so the notebook can run by itself.
The objective remains the original `OBJ(c, T, L, omega, H, GA, GB)` logic.

In [ ]:
import json
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mpl_toolkits.mplot3d import Axes3D

try:
    from scipy.ndimage import maximum_filter
    SCIPY_MAX_FILTER_AVAILABLE = True
except Exception:
    maximum_filter = None
    SCIPY_MAX_FILTER_AVAILABLE = False

import copy
from qutip import *

In [ ]:
dim = 2
GHz = 1e8
ns = 1e-9
b = destroy(dim)

s0m = tensor(b, qeye(dim))
s0p = tensor(b.dag(), qeye(dim))
s1m = tensor(qeye(dim), b)
s1p = tensor(qeye(dim), b.dag())

omega = [2*np.pi*4.2*GHz, 2*np.pi*4.7*GHz]
Delta = [2*np.pi*0*GHz, 2*np.pi*0*GHz]
Omega = [2*np.pi*1*GHz, 2*np.pi*1.1*GHz]
J = 2*np.pi*0*GHz

c_max = 10
c_min = 0
T = 10*ns
L = 2

H0 = (omega[0]*s0p*s0m + omega[1]*s1p*s1m) + \
     (1/2)*(Delta[0]*s0p*s0p*s0m*s0m + Delta[1]*s1p*s1p*s1m*s1m) + \
     J*(s0p*s1m+s0m*s1p)

Hc0m = Omega[0]*(s0m)
Hc1m = Omega[1]*(s1m)
Hc0p = Omega[0]*(s0p)
Hc1p = Omega[1]*(s1p)


def _single_gaussian_value(amp, width, center, t, T_use):
    width = float(width)
    if width == 0 or not math.isfinite(width):
        return 0.0
    # Keep the same gate placement as the original notebook objective logic.
    return float(amp) * math.exp(-((t - float(center))**2) / (width**2) * float(t >= 0 and t <= T_use))


def gaussian_channel_value(channel_params, t, T_use):
    arr = np.asarray(channel_params, dtype=float)
    if arr.ndim == 1 and arr.size >= 2:
        amp = arr[0]
        width = arr[1]
        center = arr[2] if arr.size >= 3 else T_use / 2
        return _single_gaussian_value(amp, width, center, t, T_use)
    if arr.ndim == 2 and arr.shape[1] >= 2:
        total = 0.0
        for row in arr:
            amp = row[0]
            width = row[1]
            center = row[2] if row.size >= 3 else T_use / 2
            total += _single_gaussian_value(amp, width, center, t, T_use)
        return total
    return 0.0


def C0(t, args):
    c = args['c']
    T_use = args['T']
    return gaussian_channel_value(c[0], t, T_use)


def C1(t, args):
    c = args['c']
    T_use = args['T']
    return gaussian_channel_value(c[1], t, T_use)


def C2(t, args):
    c = args['c']
    T_use = args['T']
    return gaussian_channel_value(c[2], t, T_use)


def C3(t, args):
    c = args['c']
    T_use = args['T']
    return gaussian_channel_value(c[3], t, T_use)


def ep0(t, args):
    omega_use = args['omega']
    return np.exp(1j * omega_use[0] * t)


def em0(t, args):
    omega_use = args['omega']
    return np.exp(-1j * omega_use[0] * t)


def ep1(t, args):
    omega_use = args['omega']
    return np.exp(1j * omega_use[1] * t)


def em1(t, args):
    omega_use = args['omega']
    return np.exp(-1j * omega_use[1] * t)


def C0mm(t, args):
    return 1j*C0(t, args)*ep0(t, args)


def C0pm(t, args):
    return -1j*C0(t, args)*em0(t, args)


def C0mp(t, args):
    return C1(t, args)*ep0(t, args)


def C0pp(t, args):
    return C1(t, args)*em0(t, args)


def C1mm(t, args):
    return 1j*C2(t, args)*ep1(t, args)


def C1pm(t, args):
    return -1j*C2(t, args)*em1(t, args)


def C1mp(t, args):
    return C3(t, args)*ep1(t, args)


def C1pp(t, args):
    return C3(t, args)*em1(t, args)


H = [H0, [Hc0m, C0mm], [Hc0p, C0pm], [Hc0m, C0mp], [Hc0p, C0pp],
     [Hc1m, C1mm], [Hc1p, C1pm], [Hc1m, C1mp], [Hc1p, C1pp]]

GA = sigmax()
GB = sigmax()

In [ ]:
def OBJ(c, T, L, omega, H, GA, GB):
    args = {
        'c': c,
        'T': T,
        'L': L,
        'omega': omega,
    }

    U = propagator(H, T, args=args, options=Options(nsteps=1700))

    sigma = np.sqrt((((tensor(GA, qeye(dim)).dag()*U).ptrace(1)) *
                     ((tensor(GA, qeye(dim)).dag()*U).ptrace(1)).dag()).eigenstates()[0])
    lambd = np.sqrt((((tensor(qeye(dim), GB).dag()*U).ptrace(0)) *
                     ((tensor(qeye(dim), GB).dag()*U).ptrace(0)).dag()).eigenstates()[0])

    return sum(sigma)/(2*dim**2) + sum(lambd)/(2*dim**2)


def build_H0_with_crosstalk(eta):
    eta = float(eta)
    delta_values = globals().get("Delta", [0.0, 0.0])
    if delta_values is None:
        delta_values = [0.0, 0.0]
    try:
        delta0 = delta_values[0]
        delta1 = delta_values[1]
    except Exception:
        delta0 = 0.0
        delta1 = 0.0

    H0_eta = omega[0]*s0p*s0m + omega[1]*s1p*s1m
    H0_eta = H0_eta + (1/2)*(delta0*s0p*s0p*s0m*s0m + delta1*s1p*s1p*s1m*s1m)
    H0_eta = H0_eta + eta*(s0p*s1m + s0m*s1p)
    return H0_eta


def build_H_with_crosstalk(eta):
    H0_eta = build_H0_with_crosstalk(eta)
    return [H0_eta] + H[1:]

## Stage 4 Configuration

In [ ]:
STAGE3_DIR = Path("GaussianGRAPEAnalysis_stage3_clusters_rich_gaussian_sanity_v1")
if not STAGE3_DIR.exists() and Path("Scripts/GaussianGRAPEAnalysis_stage3_clusters_rich_gaussian_sanity_v1").exists():
    STAGE3_DIR = Path("Scripts/GaussianGRAPEAnalysis_stage3_clusters_rich_gaussian_sanity_v1")

STAGE4_OUT = Path("GaussianGRAPEAnalysis_stage4_landscape_slices_rich_gaussian_sanity_v1")
if STAGE3_DIR.parts and STAGE3_DIR.parts[0] == "Scripts":
    STAGE4_OUT = Path("Scripts/GaussianGRAPEAnalysis_stage4_landscape_slices_rich_gaussian_sanity_v1")

REPRESENTATIVE_OPTIMA_PATH = STAGE3_DIR / "tables" / "representative_optima.csv"

SUCCESS_THRESHOLD = 0.999

AMP_MIN = -10.0
AMP_MAX = 10.0

T_NS_DEFAULT = 10.0
WIDTH_MIN_NS = 0.05 * T_NS_DEFAULT
WIDTH_MAX_NS = 1.50 * T_NS_DEFAULT
CENTER_MIN_NS = 0.0
CENTER_MAX_NS = T_NS_DEFAULT

DEFAULT_GRID_SIZE = 41
DEFAULT_LOCAL_AMP_SPAN_FRACTION = 0.25
DEFAULT_LOCAL_WIDTH_SPAN_FRACTION = 0.50

CHANNELS_TO_PLOT = [0]
PULSE_INDICES_TO_PLOT = [0]
MAX_REPRESENTATIVES_PER_ETA = 1
REPRESENTATIVE_MODE = "best"
USE_ONLY_HIGH_FIDELITY_CLUSTERS = True

FEATURE_COLUMNS = None

FIGURE_STATUS = []


## Utility Helpers

In [ ]:
def resolve_existing_path(path):
    path = Path(path)
    if path.exists():
        return path
    scripts_path = Path("Scripts") / path
    if scripts_path.exists():
        return scripts_path
    return path


def resolve_output_path(path, reference_dir=None):
    path = Path(path)
    if path.is_absolute():
        return path
    if path.parts and path.parts[0] == "Scripts":
        return path
    if reference_dir is not None:
        reference_dir = Path(reference_dir)
        if reference_dir.parts and reference_dir.parts[0] == "Scripts":
            return Path("Scripts") / path
    if not Path("GaussianGRAPEAnalysis_stage3_clusters_rich_gaussian_sanity_v1").exists() and Path("Scripts/GaussianGRAPEAnalysis_stage3_clusters_rich_gaussian_sanity_v1").exists():
        return Path("Scripts") / path
    return path


def ensure_stage4_dirs(out_dir):
    out_dir = Path(out_dir)
    tables_dir = out_dir / "tables"
    arrays_dir = out_dir / "arrays"
    figures_dir = out_dir / "figures"
    tables_dir.mkdir(parents=True, exist_ok=True)
    arrays_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    return tables_dir, arrays_dir, figures_dir


def safe_float(value, default=np.nan):
    try:
        if value is None:
            return default
        value = float(value)
        return value if math.isfinite(value) else default
    except Exception:
        return default


def safe_int(value, default=0):
    try:
        if value is None:
            return default
        value = float(value)
        if not math.isfinite(value):
            return default
        return int(value)
    except Exception:
        return default


def safe_bool(value, default=False):
    if isinstance(value, bool):
        return value
    if value is None:
        return default
    if isinstance(value, str):
        text = value.strip().lower()
        if text in {"true", "1", "yes", "y"}:
            return True
        if text in {"false", "0", "no", "n"}:
            return False
        return default
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass
    return bool(value)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(key): make_json_safe(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(value) for value in obj]
    if isinstance(obj, np.ndarray):
        return make_json_safe(obj.tolist())
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating, float)):
        value = float(obj)
        return value if math.isfinite(value) else None
    if isinstance(obj, Path):
        return str(obj)
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def eta_over_2pi_GHz(eta):
    eta = safe_float(eta)
    return eta / (2*np.pi*GHz) if math.isfinite(eta) else np.nan


def channel_label(channel):
    return f"C{int(channel)}"


def pulse_label(channel, pulse_index=None):
    if pulse_index is None:
        return channel_label(channel)
    return f"{channel_label(channel)} g{int(pulse_index)}"


def slice_file_stem(slice_result):
    eta_index = safe_int(slice_result["eta_index"])
    label = str(slice_result.get("cluster_label_global", f"eta_{eta_index:03d}_cluster_unknown"))
    if not label.startswith(f"eta_{eta_index:03d}_"):
        label = f"eta_{eta_index:03d}_{label}"
    label = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in label)
    pulse_index = slice_result.get("pulse_index")
    if pulse_index is None:
        return f"{label}_C{int(slice_result['channel'])}"
    return f"{label}_C{int(slice_result['channel'])}_g{int(pulse_index)}"


def save_figure(fig, path, note=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, bbox_inches="tight", dpi=160)
    FIGURE_STATUS.append({"figure": path.name, "path": str(path), "created": True, "note": note})
    plt.show()
    return path


def record_skipped_figure(name, reason):
    FIGURE_STATUS.append({"figure": name, "path": None, "created": False, "note": reason})
    print(f"Skipping {name}: {reason}")


def check_stage4_simulation_dependencies(build_H_func=None):
    required = ["OBJ", "T", "L", "omega", "GA", "GB", "ns"]
    missing = [name for name in required if name not in globals()]
    if build_H_func is None and "build_H_with_crosstalk" not in globals():
        missing.append("build_H_with_crosstalk")
    if missing:
        raise RuntimeError(
            "Stage 4 requires simulation/objective objects to be available: " + ", ".join(missing)
        )

## Representative Loading and Selection

In [ ]:
def parameter_kind(column):
    if column.startswith("A") or column.endswith("_A"):
        return "amp"
    if "width" in column:
        return "width"
    if "center" in column:
        return "center"
    if column.startswith("param_"):
        return "generic"
    return "other"


def feature_sort_key(column):
    import re
    match = re.match(r"C(\d+)_g(\d+)_(A|width_ns|center_ns)$", str(column))
    if match:
        param_order = {"A": 0, "width_ns": 1, "center_ns": 2}
        return (int(match.group(1)), int(match.group(2)), param_order[match.group(3)])
    match = re.match(r"A(\d+)$", str(column))
    if match:
        return (int(match.group(1)), 0, 0)
    match = re.match(r"width(\d+)_ns$", str(column))
    if match:
        return (int(match.group(1)), 0, 1)
    match = re.match(r"param_(\d+)$", str(column))
    if match:
        return (999, int(match.group(1)), 0)
    return (999, 999, str(column))


def infer_representative_feature_columns(df, representative_mode="best"):
    prefix = f"rep_{representative_mode}_"
    excluded = {
        "restart", "seed", "final_fidelity", "used_physical_seed",
        "count", "stop_reason", "seed_type",
    }
    feature_columns = []
    for column in df.columns:
        if not column.startswith(prefix):
            continue
        base = column[len(prefix):]
        if base in excluded:
            continue
        if parameter_kind(base) in {"amp", "width", "center", "generic"}:
            feature_columns.append(base)
    return sorted(feature_columns, key=feature_sort_key)


def set_stage4_feature_columns(feature_columns):
    global FEATURE_COLUMNS
    FEATURE_COLUMNS = list(feature_columns)
    return FEATURE_COLUMNS


def load_representative_optima(path=REPRESENTATIVE_OPTIMA_PATH):
    path = resolve_existing_path(path)
    if not path.exists():
        raise FileNotFoundError(f"Stage 3 representative optima file not found: {path}")

    df = pd.read_csv(path)
    required_eta = ["eta_index", "eta"]
    missing_eta = [col for col in required_eta if col not in df.columns]
    if missing_eta:
        raise ValueError(f"Representative table is missing required eta columns: {missing_eta}")
    if "eta_over_2pi_GHz" not in df.columns:
        df["eta_over_2pi_GHz"] = df["eta"].map(eta_over_2pi_GHz)

    available_modes = []
    feature_columns_by_mode = {}
    for mode in ["best", "centroid"]:
        feature_columns = infer_representative_feature_columns(df, representative_mode=mode)
        if feature_columns and all(f"rep_{mode}_{col}" in df.columns for col in feature_columns):
            available_modes.append(mode)
            feature_columns_by_mode[mode] = feature_columns

    if not available_modes:
        raise ValueError("No complete representative endpoint columns were found.")

    preferred_mode = REPRESENTATIVE_MODE if REPRESENTATIVE_MODE in feature_columns_by_mode else available_modes[0]
    set_stage4_feature_columns(feature_columns_by_mode[preferred_mode])

    df.attrs["input_path"] = str(path)
    df.attrs["available_representative_modes"] = available_modes
    df.attrs["feature_columns_by_mode"] = feature_columns_by_mode
    print(f"Loaded {len(df)} representative rows from {path}")
    print(f"Available representative modes: {available_modes}")
    print(f"Endpoint dimension for Stage 4: {len(FEATURE_COLUMNS)}")
    return df


def representative_fidelity_column(representative_mode):
    if representative_mode == "best":
        return "rep_best_final_fidelity"
    if representative_mode == "centroid":
        return "rep_centroid_final_fidelity"
    raise ValueError("representative_mode must be 'best' or 'centroid'.")


def select_representatives_for_slicing(
    representative_df,
    max_per_eta=1,
    use_only_high_fidelity=True,
    success_threshold=0.99,
    representative_mode="best",
):
    fid_col = representative_fidelity_column(representative_mode)
    if fid_col not in representative_df.columns:
        raise ValueError(f"Representative table is missing {fid_col}.")

    df = representative_df.copy()
    df[fid_col] = pd.to_numeric(df[fid_col], errors="coerce")
    selected_parts = []

    for eta_index, group in df.groupby("eta_index", sort=True):
        group = group.sort_values(fid_col, ascending=False).copy()
        group["selected_below_success_threshold"] = False

        if use_only_high_fidelity:
            high = group[group[fid_col] >= success_threshold].copy()
            if len(high):
                chosen = high.head(int(max_per_eta)).copy()
            else:
                chosen = group.head(int(max_per_eta)).copy()
                chosen["selected_below_success_threshold"] = True
        else:
            chosen = group.head(int(max_per_eta)).copy()
            chosen["selected_below_success_threshold"] = chosen[fid_col] < success_threshold

        selected_parts.append(chosen)

    selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else pd.DataFrame()
    selected["representative_mode"] = representative_mode
    return selected


def extract_representative_x_scaled(row, representative_mode="best"):
    if representative_mode not in {"best", "centroid"}:
        raise ValueError("representative_mode must be 'best' or 'centroid'.")
    prefix = f"rep_{representative_mode}_"
    feature_columns = FEATURE_COLUMNS or infer_representative_feature_columns(pd.DataFrame([row]), representative_mode=representative_mode)
    x_scaled = []
    missing = []
    for column in feature_columns:
        rep_column = f"{prefix}{column}"
        if rep_column not in row.index:
            missing.append(rep_column)
        else:
            x_scaled.append(safe_float(row[rep_column]))
    if missing:
        raise ValueError(f"Missing representative endpoint columns: {missing}")
    x_scaled = np.asarray(x_scaled, dtype=float)
    if x_scaled.size == 0 or not np.all(np.isfinite(x_scaled)):
        raise ValueError("Representative endpoint contains invalid values.")
    return x_scaled


## Objective Evaluation and Slice Construction

In [ ]:
def x_layout(x_scaled):
    x = np.asarray(x_scaled, dtype=float).reshape(-1)
    if x.size == 8:
        return {"n_channels": 4, "n_pulses": 1, "params_per_pulse": 2}
    if x.size % 12 == 0:
        return {"n_channels": 4, "n_pulses": x.size // 12, "params_per_pulse": 3}
    if x.size % 4 == 0:
        return {"n_channels": 4, "n_pulses": max(1, x.size // 4), "params_per_pulse": 1}
    raise ValueError(f"Unsupported endpoint dimension: {x.size}")


def coordinate_indices(x_scaled, channel, pulse_index=0):
    layout = x_layout(x_scaled)
    channel = int(channel)
    pulse_index = int(pulse_index)
    if not (0 <= channel < layout["n_channels"]):
        raise ValueError(f"channel {channel} outside layout {layout}")
    if not (0 <= pulse_index < layout["n_pulses"]):
        raise ValueError(f"pulse_index {pulse_index} outside layout {layout}")
    if layout["params_per_pulse"] == 2:
        base = 2 * channel
        return base, base + 1, None
    if layout["params_per_pulse"] == 3:
        base = channel * layout["n_pulses"] * 3 + pulse_index * 3
        return base, base + 1, base + 2
    raise ValueError(f"Unsupported layout: {layout}")


def clip_x_scaled(
    x_scaled,
    amp_bounds=(AMP_MIN, AMP_MAX),
    width_bounds_ns=(WIDTH_MIN_NS, WIDTH_MAX_NS),
    center_bounds_ns=(CENTER_MIN_NS, CENTER_MAX_NS),
):
    x = np.asarray(x_scaled, dtype=float).reshape(-1).copy()
    layout = x_layout(x)
    if layout["params_per_pulse"] == 2:
        for channel in range(4):
            a_idx, w_idx, _ = coordinate_indices(x, channel, 0)
            x[a_idx] = np.clip(x[a_idx], amp_bounds[0], amp_bounds[1])
            x[w_idx] = np.clip(x[w_idx], width_bounds_ns[0], width_bounds_ns[1])
    elif layout["params_per_pulse"] == 3:
        for channel in range(layout["n_channels"]):
            for pulse_index in range(layout["n_pulses"]):
                a_idx, w_idx, c_idx = coordinate_indices(x, channel, pulse_index)
                x[a_idx] = np.clip(x[a_idx], amp_bounds[0], amp_bounds[1])
                x[w_idx] = np.clip(x[w_idx], width_bounds_ns[0], width_bounds_ns[1])
                x[c_idx] = np.clip(x[c_idx], center_bounds_ns[0], center_bounds_ns[1])
    return x


def x_scaled_to_c_physical(x_scaled, ns_value=None):
    if ns_value is None:
        ns_value = globals().get("ns", 1.0)
    x = np.asarray(x_scaled, dtype=float).reshape(-1)
    layout = x_layout(x)
    if layout["params_per_pulse"] == 2:
        return [
            [float(x[0]), float(x[1]) * ns_value],
            [float(x[2]), float(x[3]) * ns_value],
            [float(x[4]), float(x[5]) * ns_value],
            [float(x[6]), float(x[7]) * ns_value],
        ]
    if layout["params_per_pulse"] == 3:
        c = []
        for channel in range(layout["n_channels"]):
            channel_terms = []
            for pulse_index in range(layout["n_pulses"]):
                a_idx, w_idx, c_idx = coordinate_indices(x, channel, pulse_index)
                channel_terms.append([
                    float(x[a_idx]),
                    float(x[w_idx]) * ns_value,
                    float(x[c_idx]) * ns_value,
                ])
            c.append(channel_terms)
        return c
    raise ValueError(f"Unsupported endpoint layout: {layout}")


def evaluate_gaussian_fidelity_for_eta(x_scaled, eta, build_H_func=None):
    try:
        check_stage4_simulation_dependencies(build_H_func=build_H_func)
        x_clipped = clip_x_scaled(x_scaled)
        c = x_scaled_to_c_physical(x_clipped, ns_value=globals().get("ns", 1.0))
        if build_H_func is None:
            build_H_func = globals()["build_H_with_crosstalk"]
        H_eta = build_H_func(float(eta))
        fidelity = OBJ(c, T, L, omega, H_eta, GA, GB)
        fidelity = float(np.real(fidelity))
        return fidelity if math.isfinite(fidelity) else np.nan
    except Exception:
        return np.nan


def make_local_slice_grid(
    x_center,
    channel,
    pulse_index=0,
    grid_size=41,
    amp_span_fraction=0.25,
    width_span_fraction=0.50,
    amp_bounds=(-10.0, 10.0),
    width_bounds_ns=(0.5, 15.0),
):
    x_center = clip_x_scaled(x_center, amp_bounds=amp_bounds, width_bounds_ns=width_bounds_ns)
    a_idx, w_idx, _ = coordinate_indices(x_center, channel, pulse_index)
    center_amp = float(x_center[a_idx])
    center_width = float(x_center[w_idx])

    amp_span = float(amp_span_fraction) * (amp_bounds[1] - amp_bounds[0])
    amp_min = max(amp_bounds[0], center_amp - amp_span)
    amp_max = min(amp_bounds[1], center_amp + amp_span)
    if amp_min >= amp_max:
        amp_min, amp_max = amp_bounds

    width_min = max(width_bounds_ns[0], center_width * (1 - width_span_fraction))
    width_max = min(width_bounds_ns[1], center_width * (1 + width_span_fraction))
    if width_min >= width_max:
        width_min, width_max = width_bounds_ns

    amp_values = np.linspace(amp_min, amp_max, int(grid_size))
    width_values = np.linspace(width_min, width_max, int(grid_size))
    A_grid, W_grid = np.meshgrid(amp_values, width_values)
    return amp_values, width_values, A_grid, W_grid


def evaluate_landscape_slice(
    x_center,
    eta,
    eta_index,
    cluster_label_global,
    channel,
    pulse_index=0,
    grid_size=41,
    amp_span_fraction=DEFAULT_LOCAL_AMP_SPAN_FRACTION,
    width_span_fraction=DEFAULT_LOCAL_WIDTH_SPAN_FRACTION,
    build_H_func=None,
    representative_mode="best",
    representative_fidelity=np.nan,
):
    x_center = clip_x_scaled(x_center)
    a_idx, w_idx, c_idx = coordinate_indices(x_center, channel, pulse_index)
    amp_values, width_values, A_grid, W_grid = make_local_slice_grid(
        x_center=x_center,
        channel=channel,
        pulse_index=pulse_index,
        grid_size=grid_size,
        amp_span_fraction=amp_span_fraction,
        width_span_fraction=width_span_fraction,
        amp_bounds=(AMP_MIN, AMP_MAX),
        width_bounds_ns=(WIDTH_MIN_NS, WIDTH_MAX_NS),
    )
    F_grid = np.full_like(A_grid, np.nan, dtype=float)
    failed_evaluations = 0

    for i in range(A_grid.shape[0]):
        for j in range(A_grid.shape[1]):
            x_trial = x_center.copy()
            x_trial[a_idx] = A_grid[i, j]
            x_trial[w_idx] = W_grid[i, j]
            fidelity = evaluate_gaussian_fidelity_for_eta(x_trial, eta, build_H_func=build_H_func)
            if math.isfinite(fidelity):
                F_grid[i, j] = fidelity
            else:
                failed_evaluations += 1

    center_fidelity = evaluate_gaussian_fidelity_for_eta(x_center, eta, build_H_func=build_H_func)
    if np.all(~np.isfinite(F_grid)):
        best_index = (0, 0)
        max_fidelity = np.nan
    else:
        best_flat = int(np.nanargmax(F_grid))
        best_index = np.unravel_index(best_flat, F_grid.shape)
        max_fidelity = float(F_grid[best_index])

    return {
        "eta": float(eta),
        "eta_index": int(eta_index),
        "eta_over_2pi_GHz": eta_over_2pi_GHz(eta),
        "cluster_label_global": str(cluster_label_global),
        "channel": int(channel),
        "pulse_index": int(pulse_index),
        "grid_size": int(grid_size),
        "amp_span_fraction": float(amp_span_fraction),
        "width_span_fraction": float(width_span_fraction),
        "A_grid": A_grid,
        "W_grid": W_grid,
        "F_grid": F_grid,
        "x_center": x_center,
        "center_amplitude": float(x_center[a_idx]),
        "center_width_ns": float(x_center[w_idx]),
        "center_center_ns": float(x_center[c_idx]) if c_idx is not None else float(T / (2 * ns)),
        "center_fidelity": safe_float(center_fidelity),
        "representative_fidelity": safe_float(representative_fidelity),
        "max_grid_fidelity": safe_float(max_fidelity),
        "best_grid_amplitude": safe_float(A_grid[best_index]),
        "best_grid_width_ns": safe_float(W_grid[best_index]),
        "best_grid_index": [int(best_index[0]), int(best_index[1])],
        "failed_evaluations": int(failed_evaluations),
        "representative_mode": representative_mode,
    }


## Peak Counting

In [ ]:
def manual_local_maxima_mask(F):
    F = np.asarray(F, dtype=float)
    mask = np.zeros(F.shape, dtype=bool)
    for i in range(F.shape[0]):
        for j in range(F.shape[1]):
            value = F[i, j]
            if not math.isfinite(value):
                continue
            i0 = max(0, i - 1)
            i1 = min(F.shape[0], i + 2)
            j0 = max(0, j - 1)
            j1 = min(F.shape[1], j + 2)
            neighborhood = F[i0:i1, j0:j1]
            finite_neighborhood = neighborhood[np.isfinite(neighborhood)]
            if len(finite_neighborhood) and value >= np.max(finite_neighborhood):
                mask[i, j] = True
    return mask


def count_local_peaks_2d(F_grid, threshold=None):
    if threshold is None:
        threshold = SUCCESS_THRESHOLD
    F = np.asarray(F_grid, dtype=float)
    finite_mask = np.isfinite(F)
    if not np.any(finite_mask):
        return {
            "n_peaks": 0,
            "n_high_fidelity_peaks": 0,
            "peak_indices": [],
            "peak_values": [],
            "top5_peak_indices": [],
            "top5_peak_values": [],
        }

    if SCIPY_MAX_FILTER_AVAILABLE:
        F_work = np.where(finite_mask, F, -np.inf)
        local_max = finite_mask & (F_work >= maximum_filter(F_work, size=3, mode="constant", cval=-np.inf))
    else:
        local_max = manual_local_maxima_mask(F)

    peak_indices = np.argwhere(local_max)
    peak_values = [float(F[i, j]) for i, j in peak_indices]
    order = np.argsort(peak_values)[::-1] if peak_values else []
    sorted_indices = [peak_indices[idx].tolist() for idx in order]
    sorted_values = [float(peak_values[idx]) for idx in order]

    return {
        "n_peaks": int(len(sorted_values)),
        "n_high_fidelity_peaks": int(sum(value >= threshold for value in sorted_values)),
        "peak_indices": sorted_indices,
        "peak_values": sorted_values,
        "top5_peak_indices": sorted_indices[:5],
        "top5_peak_values": sorted_values[:5],
    }

## Plotting and Slice Serialization

In [ ]:
def plot_landscape_surface(slice_result, figures_dir):
    A_grid = slice_result["A_grid"]
    W_grid = slice_result["W_grid"]
    F_grid = slice_result["F_grid"]
    stem = slice_file_stem(slice_result)
    filename = f"surface_{stem}.png"
    if not np.any(np.isfinite(F_grid)):
        record_skipped_figure(filename, "all slice fidelity values are invalid")
        return None

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")
    surface = ax.plot_surface(A_grid, W_grid, F_grid, cmap="viridis", edgecolor="none", alpha=0.9)
    fig.colorbar(surface, ax=ax, shrink=0.65, pad=0.1, label="Fidelity")

    ax.scatter(
        [slice_result["center_amplitude"]],
        [slice_result["center_width_ns"]],
        [slice_result["center_fidelity"]],
        color="red",
        s=70,
        label="representative",
    )
    ax.scatter(
        [slice_result["best_grid_amplitude"]],
        [slice_result["best_grid_width_ns"]],
        [slice_result["max_grid_fidelity"]],
        color="black",
        s=55,
        label="best grid point",
    )

    eta_index = slice_result["eta_index"]
    eta_over = slice_result["eta_over_2pi_GHz"]
    channel = slice_result["channel"]
    pulse_index = slice_result.get("pulse_index")
    label = pulse_label(channel, pulse_index)
    ax.set_xlabel(f"{label} Gaussian amplitude")
    ax.set_ylabel(f"{label} Gaussian width (ns)")
    ax.set_zlabel("Fidelity")
    ax.set_title(
        f"eta {eta_index:03d} ({eta_over:.3f}), {slice_result['cluster_label_global']}, "
        f"{label}, max={slice_result['max_grid_fidelity']:.6f}"
    )
    ax.legend()
    return save_figure(fig, Path(figures_dir) / filename)


def plot_landscape_contour(slice_result, figures_dir):
    A_grid = slice_result["A_grid"]
    W_grid = slice_result["W_grid"]
    F_grid = slice_result["F_grid"]
    stem = slice_file_stem(slice_result)
    filename = f"contour_{stem}.png"
    if not np.any(np.isfinite(F_grid)):
        record_skipped_figure(filename, "all slice fidelity values are invalid")
        return None

    fig, ax = plt.subplots(figsize=(9, 6))
    image = ax.contourf(A_grid, W_grid, F_grid, levels=30, cmap="viridis")
    fig.colorbar(image, ax=ax, label="Fidelity")
    ax.contour(A_grid, W_grid, F_grid, levels=10, colors="white", linewidths=0.5, alpha=0.6)
    ax.scatter(
        [slice_result["center_amplitude"]],
        [slice_result["center_width_ns"]],
        color="red",
        s=65,
        label="representative",
    )
    ax.scatter(
        [slice_result["best_grid_amplitude"]],
        [slice_result["best_grid_width_ns"]],
        color="black",
        s=50,
        label="best grid point",
    )

    eta_index = slice_result["eta_index"]
    eta_over = slice_result["eta_over_2pi_GHz"]
    channel = slice_result["channel"]
    pulse_index = slice_result.get("pulse_index")
    label = pulse_label(channel, pulse_index)
    ax.set_xlabel(f"{label} Gaussian amplitude")
    ax.set_ylabel(f"{label} Gaussian width (ns)")
    ax.set_title(
        f"eta {eta_index:03d} ({eta_over:.3f}), {slice_result['cluster_label_global']}, "
        f"{label}, max={slice_result['max_grid_fidelity']:.6f}"
    )
    ax.legend()
    ax.grid(True, alpha=0.2)
    return save_figure(fig, Path(figures_dir) / filename)


def save_slice_arrays(slice_result, arrays_dir):
    stem = slice_file_stem(slice_result)
    path = Path(arrays_dir) / f"slice_{stem}.npz"
    np.savez_compressed(
        path,
        A_grid=slice_result["A_grid"],
        W_grid=slice_result["W_grid"],
        F_grid=slice_result["F_grid"],
        x_center=slice_result["x_center"],
        eta=slice_result["eta"],
        eta_index=slice_result["eta_index"],
        eta_over_2pi_GHz=slice_result["eta_over_2pi_GHz"],
        channel=slice_result["channel"],
        pulse_index=slice_result.get("pulse_index", -1),
        center_fidelity=slice_result["center_fidelity"],
        max_fidelity=slice_result["max_grid_fidelity"],
    )
    return path


def flatten_slice_to_dataframe(slice_result):
    A_grid = slice_result["A_grid"]
    W_grid = slice_result["W_grid"]
    F_grid = slice_result["F_grid"]
    center_idx = np.unravel_index(
        int(np.argmin((A_grid - slice_result["center_amplitude"])**2 +
                      (W_grid - slice_result["center_width_ns"])**2)),
        A_grid.shape,
    )
    best_idx = tuple(slice_result["best_grid_index"])
    rows = []
    for i in range(A_grid.shape[0]):
        for j in range(A_grid.shape[1]):
            rows.append({
                "eta_index": slice_result["eta_index"],
                "eta": slice_result["eta"],
                "eta_over_2pi_GHz": slice_result["eta_over_2pi_GHz"],
                "cluster_label_global": slice_result["cluster_label_global"],
                "channel": slice_result["channel"],
                "pulse_index": slice_result.get("pulse_index", -1),
                "grid_i": int(i),
                "grid_j": int(j),
                "amplitude": float(A_grid[i, j]),
                "width_ns": float(W_grid[i, j]),
                "fidelity": safe_float(F_grid[i, j]),
                "is_center_nearest": bool((i, j) == center_idx),
                "is_best_grid_point": bool((i, j) == best_idx),
            })
    return pd.DataFrame(rows)

## Slice and Summary Tables

In [ ]:
def summarize_slice(slice_result, peak_info):
    top_values = peak_info.get("top5_peak_values", [])
    top_peak = safe_float(top_values[0]) if len(top_values) >= 1 else np.nan
    second_peak = safe_float(top_values[1]) if len(top_values) >= 2 else np.nan
    delta = safe_float(slice_result["max_grid_fidelity"]) - safe_float(slice_result["center_fidelity"])
    if not math.isfinite(delta):
        delta = np.nan
    distance = np.sqrt(
        (slice_result["best_grid_amplitude"] - slice_result["center_amplitude"])**2 +
        (slice_result["best_grid_width_ns"] - slice_result["center_width_ns"])**2
    )
    return {
        "eta_index": slice_result["eta_index"],
        "eta": slice_result["eta"],
        "eta_over_2pi_GHz": slice_result["eta_over_2pi_GHz"],
        "cluster_label_global": slice_result["cluster_label_global"],
        "channel": slice_result["channel"],
        "pulse_index": slice_result.get("pulse_index", -1),
        "center_amplitude": slice_result["center_amplitude"],
        "center_width_ns": slice_result["center_width_ns"],
        "center_center_ns": slice_result.get("center_center_ns", np.nan),
        "center_fidelity": slice_result["center_fidelity"],
        "representative_fidelity": slice_result["representative_fidelity"],
        "max_grid_fidelity": slice_result["max_grid_fidelity"],
        "best_grid_amplitude": slice_result["best_grid_amplitude"],
        "best_grid_width_ns": slice_result["best_grid_width_ns"],
        "delta_best_minus_center": delta,
        "distance_center_to_best": safe_float(distance),
        "n_peaks": int(peak_info["n_peaks"]),
        "n_high_fidelity_peaks": int(peak_info["n_high_fidelity_peaks"]),
        "top_peak_fidelity": top_peak,
        "second_peak_fidelity": second_peak,
        "top5_peak_values": json.dumps(make_json_safe(top_values)),
        "top5_peak_indices": json.dumps(make_json_safe(peak_info.get("top5_peak_indices", []))),
        "failed_evaluations": int(slice_result.get("failed_evaluations", 0)),
        "representative_mode": slice_result["representative_mode"],
        "grid_size": slice_result["grid_size"],
    }


def peak_count_dataframe_from_summary(slice_summary_df):
    if slice_summary_df.empty:
        return pd.DataFrame()
    cols = [
        "eta_index", "eta", "eta_over_2pi_GHz", "cluster_label_global", "channel", "pulse_index",
        "n_peaks", "n_high_fidelity_peaks", "top_peak_fidelity", "second_peak_fidelity",
    ]
    return slice_summary_df[cols].copy()

## Summary Plots and Report

In [ ]:
def plot_peak_count_vs_eta(slice_summary_df, figures_dir):
    filename = "peak_count_vs_eta.png"
    if slice_summary_df.empty:
        record_skipped_figure(filename, "slice_summary_df is empty")
        return None

    fig, ax = plt.subplots(figsize=(10, 6))
    group_keys = ["channel", "pulse_index"] if "pulse_index" in slice_summary_df.columns else ["channel"]
    for keys, group in slice_summary_df.groupby(group_keys, sort=True):
        group = group.sort_values("eta_index")
        if isinstance(keys, tuple):
            channel, pulse_index = keys
            label = pulse_label(channel, pulse_index)
        else:
            channel = keys
            label = channel_label(channel)
        ax.plot(group["eta_over_2pi_GHz"], group["n_peaks"], marker="o", label=f"{label} peaks")
        ax.plot(group["eta_over_2pi_GHz"], group["n_high_fidelity_peaks"], marker="s", linestyle="--", label=f"{label} high-fidelity peaks")

    ax.set_xlabel("Crosstalk eta / (2 pi GHz)")
    ax.set_ylabel("Peak count in 2D slice")
    ax.set_title("2D slice peak counts versus crosstalk")
    ax.grid(True, alpha=0.3)
    ax.legend()
    return save_figure(fig, Path(figures_dir) / filename)


def plot_slice_max_fidelity_vs_eta(slice_summary_df, figures_dir):
    filename = "slice_max_fidelity_vs_eta.png"
    if slice_summary_df.empty:
        record_skipped_figure(filename, "slice_summary_df is empty")
        return None

    fig, ax = plt.subplots(figsize=(10, 6))
    group_keys = ["channel", "pulse_index"] if "pulse_index" in slice_summary_df.columns else ["channel"]
    for keys, group in slice_summary_df.groupby(group_keys, sort=True):
        group = group.sort_values("eta_index")
        if isinstance(keys, tuple):
            channel, pulse_index = keys
            label = pulse_label(channel, pulse_index)
        else:
            channel = keys
            label = channel_label(channel)
        ax.plot(group["eta_over_2pi_GHz"], group["center_fidelity"], marker="o", label=f"{label} center")
        ax.plot(group["eta_over_2pi_GHz"], group["max_grid_fidelity"], marker="s", linestyle="--", label=f"{label} grid max")

    ax.axhline(SUCCESS_THRESHOLD, color="red", linestyle=":", label=f"threshold {SUCCESS_THRESHOLD}")
    ax.set_xlabel("Crosstalk eta / (2 pi GHz)")
    ax.set_ylabel("Fidelity")
    ax.set_title("Representative and local-grid max fidelity versus crosstalk")
    ax.grid(True, alpha=0.3)
    ax.legend()
    return save_figure(fig, Path(figures_dir) / filename)


def write_stage4_report(
    out_dir,
    representative_path,
    selected_representatives,
    slice_summary_df,
    validation_report,
    channels_to_plot,
    grid_size,
):
    out_dir = Path(out_dir)
    n_slices = int(len(slice_summary_df))
    failed_evals = int(slice_summary_df["failed_evaluations"].sum()) if n_slices else 0

    if n_slices:
        largest_peaks_row = slice_summary_df.loc[slice_summary_df["n_peaks"].idxmax()]
        lowest_center_row = slice_summary_df.loc[slice_summary_df["center_fidelity"].idxmin()]
        largest_improvement_row = slice_summary_df.loc[slice_summary_df["delta_best_minus_center"].idxmax()]
        high_peak_by_eta = slice_summary_df.groupby("eta_index")["n_high_fidelity_peaks"].sum()
        high_peak_note = "yes" if (high_peak_by_eta > 0).any() else "no"
    else:
        largest_peaks_row = None
        lowest_center_row = None
        largest_improvement_row = None
        high_peak_note = "insufficient data"

    lines = [
        "# Gaussian GRAPE Stage 4 Landscape Slice Report",
        "",
        f"Input representative file: `{representative_path}`",
        f"Selected representatives: {len(selected_representatives)}",
        f"Channels plotted: {list(channels_to_plot)}",
        f"Grid size: {grid_size} x {grid_size}",
        "Amplitude range strategy: centered local window clipped to [-10, 10].",
        "Width range strategy: centered multiplicative local window clipped to [0.5, 15.0] ns.",
        "Rich Gaussian strategy: when endpoints are 24D, each slice varies one selected channel/pulse component's amplitude and width while holding its center fixed.",
        f"Slices evaluated: {n_slices}",
        f"Failed objective evaluations: {failed_evals}",
        "",
        "## Key Findings",
        "",
    ]

    if largest_peaks_row is not None:
        lines.append(
            f"- Largest number of slice peaks: eta `{int(largest_peaks_row['eta_index']):03d}`, "
            f"channel C{int(largest_peaks_row['channel'])}, n_peaks={int(largest_peaks_row['n_peaks'])}."
        )
        lines.append(
            f"- Lowest center fidelity: eta `{int(lowest_center_row['eta_index']):03d}`, "
            f"channel C{int(lowest_center_row['channel'])}, center={lowest_center_row['center_fidelity']:.6f}."
        )
        lines.append(
            f"- Largest grid improvement over center: eta `{int(largest_improvement_row['eta_index']):03d}`, "
            f"channel C{int(largest_improvement_row['channel'])}, "
            f"delta={largest_improvement_row['delta_best_minus_center']:.6g}."
        )
        lines.append(f"- High-fidelity peaks persist somewhere in the selected slices: {high_peak_note}.")
    else:
        lines.append("- No slices were evaluated.")

    lines.extend([
        "",
        "## Notes",
        "",
        "- These are 2D slices through the endpoint landscape; the endpoint dimension is inferred from Stage 3 representative columns.",
        "- Peak counts are slice-specific and grid-resolution dependent.",
        "- Representative optima came from Stage 3 clustering.",
        "- Hessian analysis comes later.",
    ])

    report_path = out_dir / "stage4_report.md"
    with report_path.open("w") as f:
        f.write("\n".join(lines) + "\n")
    return report_path

## Main Stage 4 Runner

In [ ]:
def run_stage4_landscape_slices(
    stage3_dir=STAGE3_DIR,
    out_dir=STAGE4_OUT,
    channels_to_plot=CHANNELS_TO_PLOT,
    pulse_indices_to_plot=PULSE_INDICES_TO_PLOT,
    grid_size=DEFAULT_GRID_SIZE,
    max_representatives_per_eta=MAX_REPRESENTATIVES_PER_ETA,
    representative_mode=REPRESENTATIVE_MODE,
    use_only_high_fidelity_clusters=USE_ONLY_HIGH_FIDELITY_CLUSTERS,
    build_H_func=None,
):
    FIGURE_STATUS.clear()
    check_stage4_simulation_dependencies(build_H_func=build_H_func)

    stage3_dir = resolve_existing_path(stage3_dir)
    out_dir = resolve_output_path(out_dir, reference_dir=stage3_dir)
    tables_dir, arrays_dir, figures_dir = ensure_stage4_dirs(out_dir)

    rep_path = stage3_dir / "tables" / "representative_optima.csv"
    representative_df = load_representative_optima(rep_path)
    selected_reps = select_representatives_for_slicing(
        representative_df,
        max_per_eta=max_representatives_per_eta,
        use_only_high_fidelity=use_only_high_fidelity_clusters,
        success_threshold=SUCCESS_THRESHOLD,
        representative_mode=representative_mode,
    )

    grid_frames = []
    summary_rows = []
    peak_rows = []
    failures = []

    total_slices = len(selected_reps) * len(channels_to_plot) * len(pulse_indices_to_plot)
    print(f"Evaluating {total_slices} Stage 4 landscape slices.")
    print(f"Each slice uses {grid_size} x {grid_size} = {grid_size**2} objective evaluations plus center.")

    for rep_idx, row in selected_reps.iterrows():
        eta = safe_float(row["eta"])
        eta_index = safe_int(row["eta_index"])
        cluster_label_global = str(row.get("cluster_label_global", f"eta_{eta_index:03d}_cluster_unknown"))
        fid_col = representative_fidelity_column(representative_mode)
        representative_fidelity = safe_float(row.get(fid_col))

        try:
            x_center = extract_representative_x_scaled(row, representative_mode=representative_mode)
        except Exception as exc:
            failures.append({
                "eta_index": eta_index,
                "cluster_label_global": cluster_label_global,
                "reason": f"extract_representative_failed: {exc}",
            })
            continue

        for channel in channels_to_plot:
            for pulse_index in pulse_indices_to_plot:
                print(f"Slice eta {eta_index:03d}, {cluster_label_global}, C{channel}, pulse {pulse_index}")
                try:
                    slice_result = evaluate_landscape_slice(
                        x_center=x_center,
                        eta=eta,
                        eta_index=eta_index,
                        cluster_label_global=cluster_label_global,
                        channel=channel,
                        pulse_index=pulse_index,
                        grid_size=grid_size,
                        build_H_func=build_H_func,
                        representative_mode=representative_mode,
                        representative_fidelity=representative_fidelity,
                    )
                    peak_info = count_local_peaks_2d(slice_result["F_grid"], threshold=SUCCESS_THRESHOLD)
                    save_slice_arrays(slice_result, arrays_dir)
                    plot_landscape_surface(slice_result, figures_dir)
                    plot_landscape_contour(slice_result, figures_dir)
                    grid_frames.append(flatten_slice_to_dataframe(slice_result))
                    summary_row = summarize_slice(slice_result, peak_info)
                    summary_rows.append(summary_row)
                    peak_rows.append({
                        "eta_index": summary_row["eta_index"],
                        "eta": summary_row["eta"],
                        "eta_over_2pi_GHz": summary_row["eta_over_2pi_GHz"],
                        "cluster_label_global": summary_row["cluster_label_global"],
                        "channel": summary_row["channel"],
                        "pulse_index": summary_row.get("pulse_index", -1),
                        "n_peaks": summary_row["n_peaks"],
                        "n_high_fidelity_peaks": summary_row["n_high_fidelity_peaks"],
                        "top_peak_fidelity": summary_row["top_peak_fidelity"],
                        "second_peak_fidelity": summary_row["second_peak_fidelity"],
                        "top5_peak_values": summary_row["top5_peak_values"],
                        "top5_peak_indices": summary_row["top5_peak_indices"],
                    })
                except Exception as exc:
                    failures.append({
                        "eta_index": eta_index,
                        "cluster_label_global": cluster_label_global,
                        "channel": int(channel),
                        "pulse_index": int(pulse_index),
                        "reason": f"slice_failed: {exc}",
                    })

    landscape_grid_df = pd.concat(grid_frames, ignore_index=True) if grid_frames else pd.DataFrame()
    slice_summary_df = pd.DataFrame(summary_rows)
    peak_counts_df = pd.DataFrame(peak_rows)

    landscape_grid_df.to_csv(tables_dir / "landscape_grid_values.csv", index=False)
    slice_summary_df.to_csv(tables_dir / "landscape_slice_summary.csv", index=False)
    peak_counts_df.to_csv(tables_dir / "peak_counts.csv", index=False)

    plot_peak_count_vs_eta(slice_summary_df, figures_dir)
    plot_slice_max_fidelity_vs_eta(slice_summary_df, figures_dir)

    validation_report = {
        "stage": "stage4_landscape_slices",
        "stage3_dir": str(stage3_dir),
        "out_dir": str(out_dir),
        "representative_file": str(rep_path),
        "n_representatives_available": int(len(representative_df)),
        "n_representatives_selected": int(len(selected_reps)),
        "channels_to_plot": [int(channel) for channel in channels_to_plot],
        "pulse_indices_to_plot": [int(pulse_index) for pulse_index in pulse_indices_to_plot],
        "grid_size": int(grid_size),
        "n_slices_requested": int(total_slices),
        "n_slices_evaluated": int(len(slice_summary_df)),
        "n_grid_rows": int(len(landscape_grid_df)),
        "n_failures": int(len(failures)),
        "failures": failures,
        "figures": list(FIGURE_STATUS),
        "success_threshold": float(SUCCESS_THRESHOLD),
        "representative_mode": representative_mode,
        "use_only_high_fidelity_clusters": bool(use_only_high_fidelity_clusters),
    }
    with (tables_dir / "validation_report_stage4.json").open("w") as f:
        json.dump(make_json_safe(validation_report), f, indent=2)

    report_path = write_stage4_report(
        out_dir=out_dir,
        representative_path=rep_path,
        selected_representatives=selected_reps,
        slice_summary_df=slice_summary_df,
        validation_report=validation_report,
        channels_to_plot=channels_to_plot,
        grid_size=grid_size,
    )

    print(f"Stage 4 complete. Outputs saved to {out_dir}")
    print(f"Report: {report_path}")

    return {
        "selected_representatives": selected_reps,
        "landscape_grid_df": landscape_grid_df,
        "slice_summary_df": slice_summary_df,
        "peak_counts_df": peak_counts_df,
        "validation_report": validation_report,
        "report_path": report_path,
    }

## Run Stage 4

This is guarded by default because the standard run evaluates thousands of objective grid points. Set
`RUN_STAGE4_LANDSCAPE_SLICES = True` when you are ready.

In [ ]:
RUN_STAGE4_LANDSCAPE_SLICES = False

if RUN_STAGE4_LANDSCAPE_SLICES:
    stage4 = run_stage4_landscape_slices(
        stage3_dir=STAGE3_DIR,
        out_dir=STAGE4_OUT,
        channels_to_plot=[0],
        pulse_indices_to_plot=[0],
        grid_size=41,
        max_representatives_per_eta=1,
        representative_mode="best",
        use_only_high_fidelity_clusters=True,
    )

    selected_reps = stage4["selected_representatives"]
    landscape_grid_df = stage4["landscape_grid_df"]
    slice_summary_df = stage4["slice_summary_df"]
    peak_counts_df = stage4["peak_counts_df"]

    display(selected_reps.head())
    display(slice_summary_df)
    display(peak_counts_df)
else:
    print("Stage 4 landscape slices not started. Set RUN_STAGE4_LANDSCAPE_SLICES = False to run.")


# Full four-channel version. This is roughly 4x the objective-evaluation cost.
# stage4_all_channels = run_stage4_landscape_slices(
#     channels_to_plot=[0, 1, 2, 3],
#     grid_size=41,
#     max_representatives_per_eta=1,
# )